Guia ETL:

1. Duplicados
2. Valores nulos o Vacíos
3. Formato de fecha
4. Valores no válidos
5. Estandarización de texto: 

✔️ Columnas de texto: mayúsculas/minúsculas, tildes, espacios.

✔️ Aseguradoras → lista controlada ("Allianz", "La Caja", etc).

✔️ Formas de pago, tipo de comisión → validar con listas válidas

6. Verificación lógica

7. Validaciones adicionales por aseguradora

8. Estandarizar los nombres de las columnas

Opcional: Agregar una columna flag de error

Columnas Estandarizadas:

seguro
fecha_importe
tipo_producto
numero_poliza
prima
prc_comision
monto_comision
canal_venta
estado_pago

In [81]:
import pandas as pd
import openpyxl
import sys
import os

In [82]:
# Añadir la ruta de la carpeta 'functions' al sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'functions')))

In [83]:
workbook = pd.ExcelFile('../data/Comisiones.xlsx')

In [84]:
df_zurich = workbook.parse(0)
df_zurich.head(3)

,Seguro,FechaReporte,Producto,NoPóliza,Prima,Comisión (%),MontoComisión,EstadoPago,Vendedor
0,Zurich,31/10/2024,Moto,NaN,252864.09,5.0,Pendiente,Isabella Moyano Quiroga,NaN
1,Zurich,15/02/2025,Auto,wR-75659,147965.45,150.0,Pendiente,Franco Maria Belen Caceres Torres,NaN
2,Zurich,27/02/2025,Moto,NaN,26031.49,10.0,Pagado,Jeremias Morena Luna Carrizo,NaN


In [85]:
from renombrar_columnas import renombrar_cols

In [86]:
zurich_renombrar_cols = {'Seguro': 'seguro', 'FechaReporte':'fecha_reporte', 
                        'Producto':'producto', 'NoPóliza':'numero_poliza', 
                        'Prima':'prima', 'Comisión (%)':'prc_comision', 
                        'MontoComisión': 'estado_pago','EstadoPago':'canal_venta'}

df_zurich = renombrar_cols(df_zurich, zurich_renombrar_cols)

In [87]:
from duplicados import duplicados_f

In [88]:
df_zurich = duplicados_f(df_zurich)

In [89]:
df_zurich.isnull().sum()

seguro              0
fecha_reporte       0
producto            0
numero_poliza     420
prima               0
prc_comision        0
estado_pago       379
canal_venta         0
Vendedor         4000
dtype: int64

In [90]:
from nulos import tratar_nulos
df_zurich = tratar_nulos(df_zurich, {'numero_poliza': 'Sin Especificar'})

In [91]:
df_zurich.isna().sum()

seguro              0
fecha_reporte       0
producto            0
numero_poliza       0
prima               0
prc_comision        0
estado_pago       379
canal_venta         0
Vendedor         4000
dtype: int64

In [92]:
df_zurich.drop('Vendedor', axis=1, inplace=True)

In [93]:
df_zurich.isna().sum()

seguro             0
fecha_reporte      0
producto           0
numero_poliza      0
prima              0
prc_comision       0
estado_pago      379
canal_venta        0
dtype: int64

In [94]:
df_zurich = tratar_nulos(df_zurich,{'estado_pago': 'Sin Especificar'})

In [95]:
df_zurich['estado_pago'].unique()

array(['Pendiente', 'Pagado', 'Sin Especificar'], dtype=object)

In [96]:
df_zurich.head(2)

,seguro,fecha_reporte,producto,numero_poliza,prima,prc_comision,estado_pago,canal_venta
0,Zurich,31/10/2024,Moto,Sin Especificar,252864.09,5.0,Pendiente,Isabella Moyano Quiroga
1,Zurich,15/02/2025,Auto,wR-75659,147965.45,150.0,Pendiente,Franco Maria Belen Caceres Torres


In [97]:
df_zurich['monto_comision'] = round(df_zurich['prima'] * (df_zurich['prc_comision'] / 100), 2)

In [98]:
df_zurich.head(3)

,seguro,fecha_reporte,producto,numero_poliza,prima,prc_comision,estado_pago,canal_venta,monto_comision
0,Zurich,31/10/2024,Moto,Sin Especificar,252864.09,5.0,Pendiente,Isabella Moyano Quiroga,12643.20
1,Zurich,15/02/2025,Auto,wR-75659,147965.45,150.0,Pendiente,Franco Maria Belen Caceres Torres,221948.18
2,Zurich,27/02/2025,Moto,Sin Especificar,26031.49,10.0,Pagado,Jeremias Morena Luna Carrizo,2603.15
